# 軟體設計與開發 期中考 B 卷：校園咖啡廳預約取餐系統

考試時間 3 小時。請依序完成：規格書 → 產生程式 → 測試 → 交件。

## 考試規則

1. 個人獨立完成；可使用任何 AI 工具；**禁止與其他同學討論或交換檔案**。
2. 第 1 格由老師提供，**不可修改**。唯一例外：若 `openai/gpt-oss-20b` 額度用完，可將 `MODEL` 改為 `"openai/gpt-oss-120b"`，並在規格書最後註明。
3. Colab 密鑰名稱一律為 `GROQ_API_KEY`，名稱錯誤扣分。程式或輸出中出現金鑰，「交件」一項以 0 分計。
4. 測試數量：AI 解讀測試 8～10 句、流程測試 8～10 步，**不要超過上限**。
5. **為避免額度用完**，除錯時請請先"一格一格"除錯；等確認所有code cell都沒問題，在執行「重新啟動並執行全部」→上傳交卷。

## 情境

系上咖啡廳想讓同學用一句話**預約飲品、取消、查詢剩餘庫存**，並讓店長看到今天的**銷售統計**。

**初始資料：**

| 品項 | 單價（元） | 今日庫存（杯） |
|---|---|---|
| 美式咖啡 | 50 | 5 |
| 拿鐵 | 65 | 4 |
| 抹茶歐蕾 | 70 | 3 |

## 基本需求（90 分）

**B1 規格書（20 分）：** 完成筆記本中的規格書 7 節。

**B2 系統功能（40 分）：**
- 兩張資料表：菜單、訂單；每次執行都重建並放入初始資料。
- AI 從一句話中抽出「動作」與「品項」。動作有：訂購、取消、查詢。無法判斷時，回覆目前有哪些品項。
- 訂購：庫存不足時拒絕；成功時回覆金額。
- 取消：把該學生該品項最新一筆訂單的狀態改為「已取消」並釋出庫存；查不到訂單時告知使用者。
- 查詢：回覆該品項的剩餘杯數。
- 訂購與取消需要姓名，查詢不需要。
- Gradio 網頁（輸入需求與姓名，輸出系統回應）。

**B3 測試（20 分）：** AI 解讀測試 8～10 句（正常、錯誤、邊界都要有），並印出通過幾句；流程測試 8～10 步，最後顯示資料表。

**B4 交件（10 分）：** 依「交件方式」完成。

## 進階需求（30 分）

**A1 數量欄位（10 分）：** AI 再抽出第 3 個欄位「數量」（例如「兩杯拿鐵」＝2），沒說時視為 1 杯。數量不在 1～3 之間時由程式拒絕。庫存與金額改以**數量**計算。

**A2 每人上限（10 分）：** 每位學生今天最多訂 **3 杯**（所有品項加總，已取消的不計）。

**A3 統計查詢（10 分）：** 使用者說「今天各品項賣得怎樣」之類的話時，用 **JOIN 與 GROUP BY** 列出每個品項的單價、庫存、已售杯數、剩餘杯數、營收。**沒有人訂的品項也必須列出。**

## 交件方式

1. 選單「執行階段 → 重新啟動並執行全部」，確認沒有紅色錯誤，且每格執行編號連續遞增。
2. 選單「檔案 → 下載 → 下載 .ipynb」，檔名改為 `學號_期中考B.ipynb`。
3. 在電腦上建立資料夾，放入該檔案，在 GitHub 建立**公開且空的** repo（不勾選 README），然後：

```bash
git config --global user.name "你的名字"
git config --global user.email "你的 GitHub 信箱"
git init
git add 學號_期中考B.ipynb
git commit -m "期中考 B 卷"
git branch -M main
git remote add origin https://github.com/你的帳號/你的repo.git
git push -u origin main
```

4. 到作業區**同時上傳**：repo 連結、`.ipynb` 檔案本身。評分以作業區的檔案為準。

## 配分

| 層級 | 代號 | 項目 | 分數 |
|---|---|---|---|
| 基本 | B1 | 規格書 7 節完整 | 20 |
| 基本 | B2 | 系統功能 | 40 |
| 基本 | B3 | 測試與執行結果 | 20 |
| 基本 | B4 | 交件（git push＋作業區） | 10 |
| 進階 | A1 | 第 3 個欄位（數量） | 10 |
| 進階 | A2 | 每人上限規則 | 10 |
| 進階 | A3 | 統計查詢 | 10 |

# 📋 規格書（請直接在這一格修改）

學號：411322396 姓名：許紘熏 卷別：B

## 1. 系統目的:建立一個整合 Google Colab、SQLite、Groq AI 與 Gradio 介面的校園咖啡廳系統，讓使用者能透過自然語言進行餐點預約、查詢與取消。

## 2. 使用者會怎麼說（至少 5 句）
1.「我要預約一杯美式咖啡，謝謝。」
2.「幫我點 2 杯抹茶歐蕾。」
3.「我想看一下我目前預約了什麼餐點。」
4.「請幫我取消剛剛訂的拿鐵。」
5.「請問你們店裡現在還有賣什麼飲品？」

## 3. 資料表（2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| menu | item_id | INTEGER | 品項編號（Primary Key） |
| menu | name | TEXT | 品項名稱（美式咖啡、拿鐵、抹茶歐蕾） |
| menu | price | INTEGER | 單價（元） |
| menu | stock | INTEGER | 今日庫存（杯 |

| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| orders | order_id | INTEGER | 訂單編號（Primary Key, AutoIncrement） |
| orders | user_name | TEXT | 訂購人姓名 |
| orders | item_name | TEXT | 預約品項名稱 |
| orders | quantity | INTEGER | 數量 |
| orders | status | TEXT | 訂單狀態（如：已預約、已取消） |

初始資料：

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,品項,數量`
- 動作只能是：預約, 查詢, 取消, 聊天
- 品項只能是：美式咖啡, 拿鐵, 抹茶歐蕾, 無
- 數量的規則：若使用者未提及數量，預設為 1；若為查詢或聊天則輸出 0
- 無法判斷時輸出：聊天, 無, 0
- AI 不可以做的事：不可直接修改資料庫、不可捏造菜單上沒有的品項、不可回傳格式以外的文字。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 預約餐點 | 動作為「預約」且品項存在於菜單 | 成功幫您預約 [數量] 份 [品項]！ | orders 新增訂單紀錄，menu 的 stock 扣減對應數量 |
| 查詢訂單 | 動作為「查詢」 | 查詢結果：您目前預約了 [品項] 共 [數量] 杯 | 無變化（僅 SELECT 讀取） |
| 取消訂單 | 動作為「取消」且系統中有該筆訂單 | 已為您取消 [品項] 的預約。 | orders 狀態改為「已取消」，menu 的 stock 加回對應數量 |
| 一般對話 | 動作為「聊天」或無法辨識 | ［Groq AI 生成的親切回覆］ | 無變化 |

## 6. 例外情況（至少 3 個）
1.庫存不足： 使用者想預約 5 杯「抹茶歐蕾」，但庫存僅剩 3 杯，系統提示「抹茶歐蕾庫存不足，目前僅剩 3 杯」。
2.品項不存在： 使用者輸入購買菜單上沒有的商品（例如卡布奇諾），AI 識別為 無 並提示「查無此品項，請參考我們的菜單」。
3.取消不存在的訂單： 使用者要求取消未曾預約過的品項（例如從未訂過拿鐵），系統提示找不到對應的預約紀錄。

## 7. 測試清單
**7-1 AI 解讀測試（8～10 句）**

| # | 類型 | 使用者說的話 | 預期動作 | 預期品項 | 預期數量 |
|---|---|---|---|---|---|
| 1 | 正常 | 我要預約一杯美式咖啡 | 預約 | 美式咖啡 | 1 |
| 2 | 正常 | 幫我點 2 杯抹茶歐蕾 | 預約 | 抹茶歐蕾 | 2 |
| 3 | 正常 | 我想查詢我訂了什麼 | 查詢 | 無 | 0 |
| 4 | 正常 | 請取消我的拿鐵 | 取消 | 拿鐵 | 1 |
| 5 | 正常 | 請問你們有賣拿鐵嗎？ | 聊天 | 無 | 0 |
| 6 | 邊界 | 我要 3 杯拿鐵 | 預約 | 拿鐵 | 5 |
| 7 | 邊界 | 你好，今天天氣真好 | 聊天 | 無 | 0 |
| 8 | 異常 | 我要買拿鐵珍珠奶茶 | 預約 | 無 | 0 |

**7-2 流程測試（8～10 步，依序執行）**

| # | 姓名 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 許紘熏 | 幫我預約 1 杯美式咖啡 | 預約成功，美式咖啡庫存從 5 變 4 |
| 2 | 許紘熏 | 我要點 2 杯抹茶歐蕾 | 預約成功，抹茶歐蕾庫存從 3 變 1 |
| 3 | 許紘熏 | 幫我點 2 杯抹茶歐蕾 | 提示庫存不足（僅剩 1 杯），訂單未建立 |
| 4 | 許紘熏 | 查詢我的訂單 | 列出美式咖啡 1 杯、抹茶歐蕾 2 杯 |
| 5 | 許紘熏 | 取消美式咖啡 | 已取消預約，美式咖啡庫存回升為 5 |
| 6 | 許紘熏 | 查詢我的訂單 | 列表僅剩抹茶歐蕾 2 杯 |
| 7 | 許紘熏 | 我想喝珍珠奶茶 | 提示查無此品項，無法下單 |
| 8 | 許紘熏 | 謝謝你 | AI 親切回覆（如：不客氣！祝您閱讀與用餐愉快！） |

（若有改用其他模型或有測試未通過，請在此說明原因）

## 🤖 可以使用的 prompt（可自行修改）

**① 請 AI 寫規格書初稿**
```
我正在考期中考，要做一個「校園咖啡廳預約取餐系統」（Google Colab + SQLite + Groq + Gradio）。
請依照下面的題目與規格範本，幫我寫出規格書初稿。測試清單請遵守數量上限：AI 解讀測試 8～10 句、流程測試 8～10 步。
【題目】
（貼上題目）
【規格範本】
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫 Google Colab 程式，並遵守以下要求：
1. 第 1 格已經存在，請從第 2 格開始，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) AI 解讀測試 (6) 功能函式 (7) 主流程與流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經 import 了 os、re、time、sqlite3、pandas as pd、display、gradio as gr，並定義了 client、MODEL、REASONING_EFFORT、IN_COLAB，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 model=MODEL、temperature=0、reasoning_effort=REASONING_EFFORT，並用 try/except 包起來；遇到超過使用量的錯誤（429）時，等待 30 秒後重試，最多 3 次；每次呼叫後 time.sleep(2)；同一句話不要重複呼叫 AI（可以用 dict 暫存結果）。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」；數量用正規表示式找出數字，找不到就當作 1；數量是否在允許範圍內由程式檢查。
7. fetchone() 之後一定要先檢查是否為 None；資料庫連線用 try/finally 確保關閉。
8. 統計查詢要使用 JOIN 與 GROUP BY。
9. 第 5 格的測試格式為 (使用者說的話, 預期動作, 預期對象, 預期數量)，"*" 表示不在意；跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 第 8 格啟動網頁時寫成：if IN_COLAB: demo.launch(share=True)。
11. 只用第 1 格已 import 的套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [21]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [22]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
# (2) 建立資料庫與初始資料
# 每次執行都先刪除舊資料表，再重新建立 menu 與 orders 資料表，並寫入初始資料

import sqlite3

# 建立或連線至 SQLite 資料庫檔案
conn = sqlite3.connect('cafe.db')
cursor = conn.cursor()

try:
    # 刪除舊的資料表（若存在）
    cursor.execute('DROP TABLE IF EXISTS orders')
    cursor.execute('DROP TABLE IF EXISTS menu')

    # 建立菜單表 (menu)
    cursor.execute('''
        CREATE TABLE menu (
            item_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            price INTEGER NOT NULL,
            stock INTEGER NOT NULL
        )
    ''')

    # 建立訂單表 (orders)
    cursor.execute('''
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_name TEXT NOT NULL,
            item_name TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            status TEXT NOT NULL
        )
    ''')

    # 寫入初始菜單資料
    initial_menu = [
        ('美式咖啡', 50, 5),
        ('拿鐵', 65, 4),
        ('抹茶歐蕾', 70, 3)
    ]
    cursor.executemany('''
        INSERT INTO menu (name, price, stock)
        VALUES (?, ?, ?)
    ''', initial_menu)

    # 提交變更
    conn.commit()
    print("資料庫建立成功，並已成功載入初始菜單資料！")

except Exception as e:
    print(f"建立資料庫時發生錯誤：{e}")
finally:
    # 確保連線正常關閉
    conn.close()

資料庫建立成功，並已成功載入初始菜單資料！


In [23]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
# (3) 顯示資料表
# 使用 pandas 讀取並顯示目前的 menu 與 orders 資料表內容

import sqlite3
import pandas as pd

# 連線至資料庫
conn = sqlite3.connect('cafe.db')

try:
    print("=== 【目前菜單與庫存表 (menu)】 ===")
    df_menu = pd.read_sql_query("SELECT * FROM menu", conn)
    display(df_menu)

    print("\n=== 【目前訂單紀錄表 (orders)】 ===")
    df_orders = pd.read_sql_query("SELECT * FROM orders", conn)
    display(df_orders)

except Exception as e:
    print(f"讀取資料表時發生錯誤：{e}")
finally:
    conn.close()

=== 【目前菜單與庫存表 (menu)】 ===


,item_id,name,price,stock
0,1,美式咖啡,50,5
1,2,拿鐵,65,4
2,3,抹茶歐蕾,70,3



=== 【目前訂單紀錄表 (orders)】 ===


,order_id,user_name,item_name,quantity,status


In [24]:
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 品項, 數量, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
# (4) AI 解讀函式
# 透過 Groq 模型解析使用者的一句話，輸出格式為：動作, 品項, 數量
# 包含暫存機制（避免重複呼叫）以及 429 過載重試機制

ai_cache = {}

def parse_user_intent(user_input):
    # 如果同一句話已經查詢過，直接從暫存回傳，節省 API 次數
    if user_input in ai_cache:
        return ai_cache[user_input]

    # 設定系統提示詞，規範 AI 輸出格式與合法選項
    system_prompt = """
    你是一個校園咖啡廳的語意解析助手。請從使用者的一句話中萃取出三個欄位，並嚴格以逗號分隔輸出：
    格式：動作, 品項, 數量

    規則說明：
    1. 動作只能是以下四種之一：預約、查詢、取消、聊天
    2. 品項只能是以下四種之一：美式咖啡、拿鐵、抹茶歐蕾、無
    3. 數量規則：若使用者有提到數量請填入數字；若未提及數量預設為 1；若動作為查詢或聊天則數量填 0。
    4. 如果無法判斷使用者的意圖，請統一輸出：聊天, 無, 0
    5. 請絕對不要輸出額外的解釋文字或標點符號，只需輸出格式化結果。
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    response_text = ""
    max_retries = 3

    for attempt in range(max_retries):
        try:
            # 呼叫 Groq API
            completion = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0,
                reasoning_effort=REASONING_EFFORT
            )
            response_text = completion.choices[0].message.content.strip()

            # 每次呼叫後暫停 2 秒
            time.sleep(2)
            break
        except Exception as e:
            # 處理 429 速率限制錯誤，等待 30 秒後重試
            if "429" in str(e) and attempt < max_retries - 1:
                print(f"觸發 API 頻率限制 (429)，等待 30 秒後重試... (嘗試第 {attempt+1} 次)")
                time.sleep(30)
            else:
                print(f"AI 呼叫失敗：{e}")
                response_text = "聊天, 無, 0"
                break

    # 後端安全防禦：確保 AI 回覆格式正確，且品項/動作在允許清單內
    valid_actions = ["預約", "查詢", "取消", "聊天"]
    valid_items = ["美式咖啡", "拿鐵", "抹茶歐蕾", "無"]

    parts = [p.strip() for p in response_text.split(',')]

    action = parts[0] if len(parts) > 0 and parts[0] in valid_actions else "聊天"
    item = parts[1] if len(parts) > 1 and parts[1] in valid_items else "無"

    # 用正規表示式從回覆中萃取數字作為數量，找不到則預設為 1 或 0
    qty_match = re.search(r'\d+', parts[2]) if len(parts) > 2 else None
    if qty_match:
        quantity = int(qty_match.group())
    else:
        quantity = 0 if action in ["查詢", "聊天"] else 1

    result = (action, item, quantity)
    # 存入暫存字典
    ai_cache[user_input] = result
    return response_text, result

In [25]:
# ============================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# ============================================================
# (5) AI 解讀測試
# 執行 8 句測試案例（包含正常、錯誤、邊界），比對預期結果並統計通過率

test_cases = [
    ("我要預約一杯美式咖啡", "預約", "美式咖啡", 1),
    ("幫我點 2 杯抹茶歐蕾", "預約", "抹茶歐蕾", 2),
    ("我想查詢我訂了什麼", "查詢", "無", 0),
    ("請取消我的拿鐵", "取消", "拿鐵", 1),
    ("請問你們有賣拿鐵嗎？", "聊天", "無", 0),
    ("我要 3 杯拿鐵", "預約", "拿鐵", 3),
    ("你好，今天天氣真好", "聊天", "無", 0),
    ("我要買拿鐵珍珠奶茶", "預約", "無", 0)
]

passed_count = 0

print("=== 【AI 解讀測試開始】 ===\n")
for idx, (text, exp_action, exp_item, exp_qty) in enumerate(test_cases, 1):
    raw_res, (act, item, qty) = parse_user_intent(text)

    # 比對結果
    is_pass = (act == exp_action) and (item == exp_item) and (qty == exp_qty)
    if is_pass:
        passed_count += 1
        status_str = "【通過】"
    else:
        status_str = "【失敗】"

    print(f"測試 #{idx} {status_str}")
    print(弗 := f"  使用者說話：{text}")
    print(f"  AI 原始回覆：{raw_res}")
    print(f"  系統解讀結果：動作={act}, 品項={item}, 數量={qty}")
    print(f"  預期目標結果：動作={exp_action}, 品項={exp_item}, 數量={exp_qty}\n")

print(f"測試結束！總共執行 {len(test_cases)} 句，成功通過 {passed_count} 句。")

=== 【AI 解讀測試開始】 ===

測試 #1 【通過】
  使用者說話：我要預約一杯美式咖啡
  AI 原始回覆：預約, 美式咖啡, 1
  系統解讀結果：動作=預約, 品項=美式咖啡, 數量=1
  預期目標結果：動作=預約, 品項=美式咖啡, 數量=1

測試 #2 【失敗】
  使用者說話：幫我點 2 杯抹茶歐蕾
  AI 原始回覆：聊天, 無, 0
  系統解讀結果：動作=聊天, 品項=無, 數量=0
  預期目標結果：動作=預約, 品項=抹茶歐蕾, 數量=2

測試 #3 【通過】
  使用者說話：我想查詢我訂了什麼
  AI 原始回覆：查詢, 無, 0
  系統解讀結果：動作=查詢, 品項=無, 數量=0
  預期目標結果：動作=查詢, 品項=無, 數量=0

測試 #4 【通過】
  使用者說話：請取消我的拿鐵
  AI 原始回覆：取消, 拿鐵, 1
  系統解讀結果：動作=取消, 品項=拿鐵, 數量=1
  預期目標結果：動作=取消, 品項=拿鐵, 數量=1

測試 #5 【失敗】
  使用者說話：請問你們有賣拿鐵嗎？
  AI 原始回覆：查詢, 拿鐵, 0
  系統解讀結果：動作=查詢, 品項=拿鐵, 數量=0
  預期目標結果：動作=聊天, 品項=無, 數量=0

測試 #6 【失敗】
  使用者說話：我要 3 杯拿鐵
  AI 原始回覆：聊天, 無, 0
  系統解讀結果：動作=聊天, 品項=無, 數量=0
  預期目標結果：動作=預約, 品項=拿鐵, 數量=3

測試 #7 【通過】
  使用者說話：你好，今天天氣真好
  AI 原始回覆：聊天, 無, 0
  系統解讀結果：動作=聊天, 品項=無, 數量=0
  預期目標結果：動作=聊天, 品項=無, 數量=0

測試 #8 【失敗】
  使用者說話：我要買拿鐵珍珠奶茶
  AI 原始回覆：聊天, 無, 0
  系統解讀結果：動作=聊天, 品項=無, 數量=0
  預期目標結果：動作=預約, 品項=無, 數量=0

測試結束！總共執行 8 句，成功通過 4 句。


In [34]:
# ============================================================
# 第 6 格：功能函式
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線；統計用 JOIN＋GROUP BY
# ============================================================
# (6) 功能函式
# 實作系統的核心商業邏輯：預約、查詢、取消，並嚴格控制資料庫交易與防呆檢查

import sqlite3

def handle_user_request(user_name, user_input):
    # 1. 直接呼叫第 4 格的 AI 函式取得解析結果
    # 這裡直接用變數接好，避免解包錯誤
    ai_res = parse_user_intent(user_input)
    if isinstance(ai_res, tuple) and len(ai_res) == 2:
        raw_res, (action, item, quantity) = ai_res
    else:
        action, item, quantity = "聊天", "無", 0

    # 檢查是否需要姓名（預約與取消需要姓名）
    if action in ["預約", "取消"] and (not user_name or user_name.strip() == ""):
        return "【系統提示】訂購與取消功能必須輸入學生姓名，請填寫後再試一次！"

    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()

    try:
        # --- 情況 A：預約（訂購）---
        if action == "預約":
            if item == "無":
                return "【系統提示】查無此品項，請參考我們的菜單（美式咖啡、拿鐵、抹茶歐蕾）。"

            if quantity <= 0:
                return "【系統提示】訂購數量必須大於 0。"

            # 檢查庫存與單價
            cursor.execute("SELECT price, stock FROM menu WHERE name = ?", (item,))
            row = cursor.fetchone()

            if row is None:
                return "【系統提示】查無此品項。"

            price, stock = row[0], row[1]

            if stock < quantity:
                return f"【系統提示】訂購失敗，{item} 庫存不足（目前僅剩 {stock} 杯）。"

            # 扣減庫存
            new_stock = stock - quantity
            cursor.execute("UPDATE menu SET stock = ? WHERE name = ?", (new_stock, item))

            # 新增訂單記錄
            cursor.execute("""
                INSERT INTO orders (user_name, item_name, quantity, status)
                VALUES (?, ?, ?, '已預約')
            """, (user_name.strip(), item, quantity))

            conn.commit()
            total_price = price * quantity
            return f"成功訂購！品項：{item}，數量：{quantity} 杯，總金額為：{total_price} 元。"

        # --- 情況 B：查詢庫存 ---
        elif action == "查詢":
            if item == "無":
                # 若未指定特定品項，回報所有品項的剩餘庫存
                cursor.execute("SELECT name, stock FROM menu")
                rows = cursor.fetchall()
                menu_status = "、".join([f"{r[0]}剩餘 {r[1]} 杯" for r in rows])
                return f"【庫存查詢結果】目前各品項庫存：{menu_status}"
            else:
                cursor.execute("SELECT stock FROM menu WHERE name = ?", (item,))
                row = cursor.fetchone()
                if row is None:
                    return f"【系統提示】查無品項「{item}」。"
                stock = row[0]
                return f"【庫存查詢結果】{item} 目前剩餘杯數為 {stock} 杯。"

        # --- 情況 C：取消訂單 ---
        elif action == "取消":
            if item == "無":
                return "【系統提示】請指定您想要取消的品項名稱。"

            # 尋找該學生該品項「最新一筆」狀態為「已預約」的訂單
            cursor.execute("""
                SELECT order_id, quantity FROM orders
                WHERE user_name = ? AND item_name = ? AND status = '已預約'
                ORDER BY order_id DESC LIMIT 1
            """, (user_name.strip(), item))

            order_row = cursor.fetchone()
            if order_row is None:
                return f"【系統提示】查無 {user_name.strip()} 訂購「{item}」的有效訂單紀錄。"

            order_id, ordered_qty = order_row[0], order_row[1]

            # 將訂單狀態改為「已取消」
            cursor.execute("UPDATE orders SET status = '已取消' WHERE order_id = ?", (order_id,))

            # 把庫存加回 menu 表
            cursor.execute("UPDATE menu SET stock = stock + ? WHERE name = ?", (ordered_qty, item))

            conn.commit()
            return f"已為您取消 {user_name.strip()} 最新一筆 {item} 訂單（共 {ordered_qty} 杯），並已補回庫存。"

        # --- 情況 D：聊天 / 無法判斷 ---
        else:
            # 讀取目前菜單品項回覆給使用者
            cursor.execute("SELECT name FROM menu")
            items = [r[0] for r in cursor.fetchall()]
            item_list_str = "、".join(items)
            return f"無法判斷您的需求。目前咖啡廳提供的品項有：{item_list_str}。"

    except Exception as e:
        conn.rollback()
        return f"系統執行發生異常：{e}"
    finally:
        conn.close()

In [35]:
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ============================================================
# (7) 主流程與流程測試
# 依序執行 8 步流程測試，驗證完整業務情境，最後顯示資料庫統計結果

import sqlite3
import pandas as pd

# 每次執行測試前，重新初始化資料庫與初始資料（確保測試環境乾淨）
def reset_database():
    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    cursor.execute('DROP TABLE IF EXISTS orders')
    cursor.execute('DROP TABLE IF EXISTS menu')
    cursor.execute('CREATE TABLE menu (item_id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL, price INTEGER NOT NULL, stock INTEGER NOT NULL)')
    cursor.execute('CREATE TABLE orders (order_id INTEGER PRIMARY KEY AUTOINCREMENT, user_name TEXT NOT NULL, item_name TEXT NOT NULL, quantity INTEGER NOT NULL, status TEXT NOT NULL)')

    initial_menu = [('美式咖啡', 50, 5), ('拿鐵', 65, 4), ('抹茶歐蕾', 70, 3)]
    cursor.executemany('INSERT INTO menu (name, price, stock) VALUES (?, ?, ?)', initial_menu)
    conn.commit()
    conn.close()

# 重設資料庫
reset_database()

# 定義 8 步流程測試清單
workflow_tests = [
    (1, "許紘熏", "幫我預約 1 杯美式咖啡"),
    (2, "許紘熏", "我要點 2 杯抹茶歐蕾"),
    (3, "許紘熏", "幫我點 2 杯抹茶歐蕾"),
    (4, "", "查詢抹茶歐蕾剩幾杯"),
    (5, "許紘熏", "我想取消美式咖啡"),
    (6, "", "查詢美式咖啡庫存"),
    (7, "許紘熏", "取消拿鐵"),
    (8, "許紘熏", "想買便當")
]

print("=== 【開始執行 8 步流程測試】 ===\n")
for step_num, name, text in workflow_tests:
    print(f"--- 步驟 #{step_num} ---")
    print(f"輸入姓名：{name if name else '(無)'}")
    print(f"使用者說：{text}")
    response = handle_user_request(name, text)
    print(f"系統回覆：{response}\n")

# 最後顯示最終的資料庫資料表內容
print("=== 【測試結束後：資料表最終狀態】 ===")
conn = sqlite3.connect('cafe.db')
try:
    print("\n--- 菜單與剩餘庫存 ---")
    display(pd.read_sql_query("SELECT * FROM menu", conn))

    print("\n--- 所有訂單紀錄 ---")
    display(pd.read_sql_query("SELECT * FROM orders", conn))

    print("\n--- 【店長銷售統計】（使用 JOIN 與 GROUP BY） ---")
    # 已修正：補上 SELECT 關鍵字
    df_sales = pd.read_sql_query("""
        SELECT
            m.name AS 品項名稱,
            COALESCE(SUM(o.quantity), 0) AS 總銷售杯數,
            COALESCE(SUM(o.quantity * m.price), 0) AS 總銷售金額
        FROM menu m
        LEFT JOIN orders o ON m.name = o.item_name AND o.status = '已預約'
        GROUP BY m.name
    """, conn)
    display(df_sales)
finally:
    conn.close()

=== 【開始執行 8 步流程測試】 ===

--- 步驟 #1 ---
輸入姓名：許紘熏
使用者說：幫我預約 1 杯美式咖啡
系統回覆：無法判斷您的需求。目前咖啡廳提供的品項有：美式咖啡、拿鐵、抹茶歐蕾。

--- 步驟 #2 ---
輸入姓名：許紘熏
使用者說：我要點 2 杯抹茶歐蕾
系統回覆：無法判斷您的需求。目前咖啡廳提供的品項有：美式咖啡、拿鐵、抹茶歐蕾。

--- 步驟 #3 ---
輸入姓名：許紘熏
使用者說：幫我點 2 杯抹茶歐蕾
系統回覆：無法判斷您的需求。目前咖啡廳提供的品項有：美式咖啡、拿鐵、抹茶歐蕾。

--- 步驟 #4 ---
輸入姓名：(無)
使用者說：查詢抹茶歐蕾剩幾杯
系統回覆：無法判斷您的需求。目前咖啡廳提供的品項有：美式咖啡、拿鐵、抹茶歐蕾。

--- 步驟 #5 ---
輸入姓名：許紘熏
使用者說：我想取消美式咖啡
系統回覆：無法判斷您的需求。目前咖啡廳提供的品項有：美式咖啡、拿鐵、抹茶歐蕾。

--- 步驟 #6 ---
輸入姓名：(無)
使用者說：查詢美式咖啡庫存
系統回覆：無法判斷您的需求。目前咖啡廳提供的品項有：美式咖啡、拿鐵、抹茶歐蕾。

--- 步驟 #7 ---
輸入姓名：許紘熏
使用者說：取消拿鐵
系統回覆：無法判斷您的需求。目前咖啡廳提供的品項有：美式咖啡、拿鐵、抹茶歐蕾。

--- 步驟 #8 ---
輸入姓名：許紘熏
使用者說：想買便當
系統回覆：無法判斷您的需求。目前咖啡廳提供的品項有：美式咖啡、拿鐵、抹茶歐蕾。

=== 【測試結束後：資料表最終狀態】 ===

--- 菜單與剩餘庫存 ---


,item_id,name,price,stock
0,1,美式咖啡,50,5
1,2,拿鐵,65,4
2,3,抹茶歐蕾,70,3



--- 所有訂單紀錄 ---


,order_id,user_name,item_name,quantity,status



--- 【店長銷售統計】（使用 JOIN 與 GROUP BY） ---


,品項名稱,總銷售杯數,總銷售金額
0,抹茶歐蕾,0,0
1,拿鐵,0,0
2,美式咖啡,0,0


In [36]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================
# (8) Gradio 網頁
# 建立網頁介面，提供姓名輸入框、需求輸入框與系統回應輸出框

import gradio as gr

def cafe_chatbot(user_name, user_input):
    if not user_input or user_input.strip() == "":
        return "請輸入您的需求！"
    # 呼叫核心功能函式處理
    response = handle_user_request(user_name, user_input)
    return response

# 建構 Gradio 介面
demo = gr.Interface(
    fn=cafe_chatbot,
    inputs=[
        gr.Textbox(label="學生姓名 (訂購或取消時必填，查詢可留空)", placeholder="例如：許紘熏"),
        gr.Textbox(label="您的需求 / 想說的話", placeholder="例如：我要訂購一杯美式咖啡")
    ],
    outputs=gr.Textbox(label="系統回覆"),
    title="校園咖啡廳預約取餐系統",
    description="歡迎使用校園咖啡廳！您可以透過自然語言進行美式咖啡、拿鐵、抹茶歐蕾的訂購、取消與庫存查詢。"
)

# 在 Google Colab 環境中啟動公開網頁連結
if IN_COLAB:
    demo.launch(share=True)
else:
    demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23aabbde642fc94b86.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ✅ 交件前檢查清單

- [ ] 規格書 7 節都填完；測試數量沒有超過上限
- [ ] 程式與輸出中**沒有**任何 `gsk_` 開頭的金鑰
- [ ] 「執行階段 → 重新啟動並執行全部」，全部跑完沒有紅色錯誤
- [ ] 每一格左邊的執行編號連續遞增（1、2、3…）
- [ ] **跑完之後**才「檔案 → 下載 → 下載 .ipynb」，檔名為 `學號_期中考B.ipynb`
- [ ] git push 到**公開**的 repo，用無痕視窗確認打得開
- [ ] 作業區**同時上傳** repo 連結與 .ipynb 檔案